# TripoSR on Colab
画像1枚 → 3Dメッシュ（.glb）を無料GPUで生成するノートブック。

使い方：上から順に全部のセルを実行 → 画像アップロード → 最後に .glb をダウンロード。
- TripoSR: MIT license (VAST-AI-Research/TripoSR), weights: stabilityai/TripoSR
- 無料枠の T4 GPU で十分回る（VRAM 6GB級でOK）

In [ ]:
# 1) セットアップ（数分かかる）
# torchmcubes は build isolation だと torch を見つけられず落ちるので分離して入れる
import subprocess, sys
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!sed -i '/torchmcubes/d' requirements.txt
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation git+https://github.com/tatsy/torchmcubes.git
!pip install -q onnxruntime trimesh
print('setup done')

In [ ]:
# 2) 画像をアップロード
from google.colab import files
up = files.upload()
img_path = list(up.keys())[0]
print(img_path)

In [ ]:
# 3) 前処理：三面図なら FRONT だけ切り抜く（単体画像なら CROP=None のまま）
from PIL import Image
CROP = (0.02, 0.06, 0.34, 0.97)  # (left, top, right, bottom) 0..1 / None で無効
im = Image.open(img_path).convert('RGB')
if CROP:
    w, h = im.size
    im = im.crop((int(CROP[0]*w), int(CROP[1]*h), int(CROP[2]*w), int(CROP[3]*h)))
im.save('input.png')
im

In [ ]:
# 4) 推論（重みは初回だけ自動ダウンロード、約1.6GB）
!python run.py input.png --output-dir out
import glob
print(glob.glob('out/**/*.obj', recursive=True))

In [ ]:
# 5) .glb に変換してダウンロード
import glob, trimesh
obj = sorted(glob.glob('out/**/*.obj', recursive=True))[0]
scene = trimesh.load(obj)
scene.export('mesh.glb')
from google.colab import files
files.download('mesh.glb')
print('downloaded mesh.glb')

## メモ
- 無料枠はセッションが切れると全部消える。生成したらすぐダウンロードすること
- 顔や刺繍などの細部は潰れがち。背面はAIの補完（捏造）
- 仕上げは Blender 側（krimskrams/models の build_model.py 流儀）で行う